# 04 · EDA Pantanal

Análise exploratória do **Pantanal** (2020–2023), seguindo as subperguntas do README:

1. Volume · 2. Sazonalidade · 3. Anos atípicos · 4. Clima × fogo · 5. Intensidade (FRP) · mapa e cobertura.

**Pré-requisito:** rodar antes o notebook 03 (gera `focos_com_clima.parquet` e `painel_estacao_dia.parquet` no Drive). O notebook 04 (Pantanal) e o 05 (Cerrado) são idênticos, só muda a variável `BIOMA` na primeira célula. As figuras são salvas em `MyDrive/figuras/` para usar nos slides.

Bioma de cada foco = campo `Bioma` do INPE. Clima = INMET (estações a até 100 km do bioma).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from google.colab import drive

drive.mount('/content/drive')

BIOMA = 'Pantanal'   # único parâmetro que muda entre os notebooks 04 e 05

RAIO_KM = 100   # raio principal da junção (README)
BASE     = '/content/drive/MyDrive'
F_FOCOS  = f'{BASE}/datasets_queimadas/focos_com_clima.parquet'      # saída do notebook 03
F_PAINEL = f'{BASE}/datasets_queimadas/painel_estacao_dia.parquet'   # saída do notebook 03
SHP_ZIP  = f'{BASE}/datasets_inmet/dmRgNtTrBmBr_v2025_vetor_e250K.zip'  # ajuste se o zip do IBGE estiver em outra pasta
FIG_DIR  = f'{BASE}/figuras'
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
COR = {'Pantanal': '#2a9d8f', 'Cerrado': '#e76f51'}
MESES = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']


def salvar(fig, nome):
    fig.savefig(f'{FIG_DIR}/{nome}.png', dpi=150, bbox_inches='tight')


# Área dos biomas: calculada dos polígonos do IBGE (projeção de área equivalente).
# Se o shapefile não puder ser lido, usa valores de referência do IBGE (km²).
AREA_REF = {'Pantanal': 150355.0, 'Cerrado': 2036448.0}

def areas_biomas():
    try:
        import geopandas as gpd
        g = gpd.read_file(f'zip://{SHP_ZIP}!dmRgNtTrBmBr_v2025_dominios_vetor_e250K.shp')
        g = g.to_crs('ESRI:102033')                       # Albers América do Sul (área equivalente)
        g['km2'] = g.geometry.area / 1e6
        a = g.groupby('nm_bm')['km2'].sum()
        out = {b: float(a[a.index.str.lower() == b.lower()].iloc[0]) for b in AREA_REF}
        print('Área calculada dos polígonos do IBGE (km²):', {k: round(v) for k, v in out.items()})
        return out, g
    except Exception as e:
        print(f'Não consegui ler o shapefile ({type(e).__name__}); usando áreas de referência do IBGE:', AREA_REF)
        return AREA_REF, None

AREA, GBIOMAS = areas_biomas()


## Carga dos dados

In [ ]:
focos_all  = pd.read_parquet(F_FOCOS)
painel_all = pd.read_parquet(F_PAINEL)
for d in (focos_all, painel_all):
    d['Data'] = pd.to_datetime(d['Data'])

f = focos_all[focos_all['Bioma'] == BIOMA].copy()
f['Ano'] = f['Data'].dt.year
f['Mes'] = f['Data'].dt.month

# Painel: estações a até 100 km do bioma; contagem = focos DESTE bioma em volta da estação (bioma do foco)
col_dist = f'dist_{BIOMA.lower()}_km'
p = painel_all[painel_all[col_dist] <= RAIO_KM].copy()
p['Ano'] = p['Data'].dt.year
p['Mes'] = p['Data'].dt.month
COL100, COL50 = f'n_focos_r100_{BIOMA}', f'n_focos_r50_{BIOMA}'
p['Teve_foco'] = p[COL100] > 0

print(f'{BIOMA}: {len(f):,} focos ({f.Data.min().date()} a {f.Data.max().date()})')
print(f'Painel: {p["Codigo"].nunique()} estações a até {RAIO_KM} km do bioma, {len(p):,} estação-dias, '
      f'{p["Teve_foco"].mean():.1%} com algum foco em volta')


## 1. Volume

In [ ]:
# --- Subpergunta 1: volume por ano ---
vol = f.groupby('Ano').size().rename('focos').to_frame()
vol['focos_por_1000km2'] = (vol['focos'] / AREA[BIOMA] * 1000).round(2)
display(vol)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(vol.index.astype(str), vol['focos'], color=COR[BIOMA])
for x, v in zip(vol.index.astype(str), vol['focos']):
    ax.text(x, v, f'{v:,}'.replace(',', '.'), ha='center', va='bottom')
ax.set_title(f'Focos de calor por ano · {BIOMA}')
ax.set_ylabel('Focos (satélite AQUA_M-T)')
ax.set_xlabel('')
salvar(fig, f'{BIOMA.lower()}_01_volume_anual')
plt.show()


## 2. Sazonalidade

In [ ]:
# --- Subpergunta 2: sazonalidade ---
mapa = f.groupby(['Ano', 'Mes']).size().unstack(fill_value=0).reindex(columns=range(1, 13), fill_value=0)
mapa.columns = MESES

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1.5, 1]})
sns.heatmap(mapa, annot=True, fmt='d', cmap='YlOrRd', cbar=False, ax=ax[0])
ax[0].set_title(f'Focos por mês e ano · {BIOMA}')
ax[0].set_ylabel('')

pct_mes = (mapa.sum() / mapa.sum().sum() * 100)
ax[1].bar(MESES, pct_mes.values, color=COR[BIOMA])
ax[1].set_title('% dos focos do período em cada mês')
ax[1].set_ylabel('%')
plt.tight_layout()
salvar(fig, f'{BIOMA.lower()}_02_sazonalidade')
plt.show()

top3 = pct_mes.sort_values(ascending=False).head(3)
print('Meses com mais focos (% do total):', ', '.join(f'{m} ({v:.1f}%)' for m, v in top3.items()))


## 3. Anos atípicos

In [ ]:
# --- Subpergunta 3: anos atípicos ---
anual = f.groupby('Ano').size()
ano_pico = anual.idxmax()
razao = anual.max() / anual.drop(ano_pico).median()
print(f'Ano de pico: {ano_pico} com {anual.max():,} focos = {razao:.1f}x a mediana dos outros anos.')
print(f'Participação do ano de pico no total do período: {anual.max() / anual.sum():.0%}')

fig, ax = plt.subplots(figsize=(9, 4.2))
for ano in sorted(f['Ano'].unique()):
    s = mapa.loc[ano]
    ax.plot(MESES, s.values, marker='o', linewidth=3 if ano == ano_pico else 1.5,
            alpha=1 if ano == ano_pico else 0.7, label=str(ano))
ax.set_title(f'Focos por mês, cada ano · {BIOMA} (ano de pico em destaque)')
ax.set_ylabel('Focos')
ax.legend(title='Ano')
salvar(fig, f'{BIOMA.lower()}_03_anos_atipicos')
plt.show()


## 4. Intensidade (FRP)

In [ ]:
# --- Subpergunta 5: intensidade (FRP, potência radiativa do fogo, em MW) ---
fr = f.dropna(subset=['FRP'])
print(f'Focos com FRP informado: {len(fr) / len(f):.1%}')
display(fr.groupby('Ano')['FRP'].describe(percentiles=[.5, .9]).round(1)[['count', 'mean', '50%', '90%', 'max']])

fig, ax = plt.subplots(figsize=(8, 4.2))
sns.boxplot(data=fr, x='Ano', y='FRP', color=COR[BIOMA], fliersize=1, ax=ax)
ax.set_yscale('log')
ax.set_title(f'Intensidade do fogo (FRP) por ano · {BIOMA}')
ax.set_ylabel('FRP (MW, escala log)')
salvar(fig, f'{BIOMA.lower()}_04_frp')
plt.show()


## 5. Clima × fogo

In [ ]:
# --- Subpergunta 4: clima × fogo (dados do INMET, estações a até 100 km) ---
VARS = {'Dias_sem_chuva': 'Dias sem chuva', 'Umid_min': 'Umidade mínima (%)',
        'Temp_max': 'Temperatura máxima (°C)', 'Precip_total': 'Chuva do dia (mm)'}

fig, axs = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (v, nome) in zip(axs, VARS.items()):
    sns.boxplot(data=p, x='Teve_foco', y=v, hue='Teve_foco', palette=['#bbbbbb', COR[BIOMA]],
                showfliers=False, legend=False, ax=ax)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['sem foco', 'com foco'])
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title(nome)
plt.suptitle(f'Dias com e sem foco em volta da estação · {BIOMA}', y=1.03)
plt.tight_layout()
salvar(fig, f'{BIOMA.lower()}_05_clima_dias_com_sem_foco')
plt.show()

print('Medianas (dias sem foco vs com foco):')
display(p.groupby('Teve_foco')[list(VARS)].median().round(1).rename(index={False: 'sem foco', True: 'com foco'}))


In [ ]:
# Probabilidade de haver foco por faixa de seca/umidade
p['Faixa_seca'] = pd.cut(p['Dias_sem_chuva'], [-1, 0, 6, 14, 29, 59, 10_000],
                         labels=['0', '1–6', '7–14', '15–29', '30–59', '60+'])
p['Faixa_umid'] = pd.cut(p['Umid_min'], [-1, 20, 30, 40, 60, 101],
                         labels=['<20', '20–30', '30–40', '40–60', '>60'])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col, tit in [(ax[0], 'Faixa_seca', 'Dias sem chuva (INMET)'), (ax[1], 'Faixa_umid', 'Umidade mínima do dia (%)')]:
    taxa = p.groupby(col, observed=True)['Teve_foco'].mean() * 100
    a.bar(taxa.index.astype(str), taxa.values, color=COR[BIOMA])
    a.set_title(f'% de dias com foco, por faixa de {tit}')
    a.set_ylabel('% de estação-dias com foco')
plt.tight_layout()
salvar(fig, f'{BIOMA.lower()}_06_probabilidade_foco_por_faixa')
plt.show()

# Correlação de Spearman entre nº de focos em volta da estação e cada variável (100 km e teste de 50 km)
linhas = []
for v, nome in VARS.items():
    for col, raio in [(COL100, '100 km'), (COL50, '50 km')]:
        d = p[[v, col]].dropna()
        rho = spearmanr(d[v], d[col])[0]
        linhas.append({'variável': nome, 'raio': raio, 'rho_spearman': round(rho, 3), 'n': len(d)})
display(pd.DataFrame(linhas).pivot(index='variável', columns='raio', values='rho_spearman'))
print('Leitura: rho positivo = mais focos quando a variável é maior; negativo = mais focos quando é menor.')
print('Se 100 km e 50 km dão o mesmo sinal e magnitude parecida, o resultado é robusto ao raio.')


## 6. Mapa e cobertura das estações

In [ ]:
# --- Mapa: onde estão os focos e as estações ---
amostra = f.sample(min(len(f), 40_000), random_state=0)
est_b = p.drop_duplicates('Codigo')

fig, ax = plt.subplots(figsize=(7, 7))
if GBIOMAS is not None:
    gb = GBIOMAS.to_crs('EPSG:4326')
    gb[gb['nm_bm'].str.lower() == BIOMA.lower()].boundary.plot(ax=ax, color='k', linewidth=1)
ax.scatter(amostra['Longitude'], amostra['Latitude'], s=1, alpha=0.25, color=COR[BIOMA], label='focos (amostra)')
ax.scatter(est_b['Longitude'], est_b['Latitude'], s=18, marker='^', color='navy', label='estações INMET (≤100 km)')
ax.set_title(f'{BIOMA}: focos e estações'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(loc='lower left'); ax.set_aspect('equal')
salvar(fig, f'{BIOMA.lower()}_07_mapa')
plt.show()

cob = focos_all[focos_all['Bioma'] == BIOMA]
print(f'Cobertura: {cob["Tem_clima"].mean():.1%} dos focos têm estação a até {RAIO_KM} km no dia; '
      f'distância mediana até a estação mais próxima com dado: {cob["Dist_estacao_km"].median():.0f} km.')


## Conclusões do Pantanal

*(Escreva aqui 4 ou 5 frases com os números que saíram acima: volume e ano de pico, meses de pico, relação com dias sem chuva/umidade, intensidade do FRP e a ressalva da cobertura de estações. Use as saídas das células como base.)*